# Data Utilities (HGBC Variant)
Load the Wisconsin Diagnostic Breast Cancer dataset, discretize numeric features (8 selected), and apply random feature masking to simulate partial observability.

Differences from NB model:
- Uses 8 selected features instead of all 30
- Includes `impute_mode()` helper (used by DT attempt, not needed for HGBC)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath(".."), "src"))

import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

## Selected Features
8 features chosen for discriminative power and low redundancy.

In [ ]:
SELECTED_FEATURES = [11, 14, 17, 18, 21, 22, 24, 28]

## Load Breast Cancer Dataset

In [ ]:
def load_breast_cancer_data(n_bins=10):
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    X_raw = data.data[:, SELECTED_FEATURES]
    try:
        disc = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy="quantile",
                                quantile_method="averaged_inverted_cdf")
    except TypeError:
        disc = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy="quantile")
    X = disc.fit_transform(X_raw).astype(int)
    y = data.target
    return X, y

## Apply Random Mask

In [ ]:
def apply_mask(X, missing_rate, rng):
    X_masked = X.astype(float).copy()
    n_rows, n_cols = X_masked.shape
    total_entries = n_rows * n_cols
    n_missing = int(total_entries * missing_rate)

    flat_indices = rng.choice(total_entries, size=n_missing, replace=False)
    row_idx, col_idx = np.unravel_index(flat_indices, (n_rows, n_cols))

    mask = np.zeros((n_rows, n_cols), dtype=bool)
    for r, c in zip(row_idx, col_idx):
        X_masked.iat[r, c] = np.nan
        mask[r, c] = True

    return X_masked, mask

## Mode Imputation (for DT baseline, not used by HGBC)

In [ ]:
def impute_mode(X):
    X_imp = X.copy()
    for j in range(X_imp.shape[1]):
        col = X_imp[:, j]
        nan_mask = np.isnan(col)
        if not nan_mask.any():
            continue
        observed = col[~nan_mask]
        if len(observed) > 0:
            vals, counts = np.unique(observed.astype(int), return_counts=True)
            mode_val = vals[np.argmax(counts)]
            X_imp[nan_mask, j] = mode_val
        else:
            X_imp[nan_mask, j] = 0
    return X_imp.astype(int)

## Demo

In [ ]:
X, y = load_breast_cancer_data()
print(f"Breast Cancer (Wisconsin): {X.shape[0]} instances, {X.shape[1]} features")
print(f"  Class distribution: {np.bincount(y)}  (0=malignant, 1=benign)")

rng = np.random.default_rng(42)
X_m50, mask50 = apply_mask(pd.DataFrame(X), 0.50, rng)
rng2 = np.random.default_rng(42)
X_m70, mask70 = apply_mask(pd.DataFrame(X), 0.70, rng2)
print(f"\n50% mask: {mask50.sum()} entries ({mask50.mean()*100:.1f}%)")
print(f"70% mask: {mask70.sum()} entries ({mask70.mean()*100:.1f}%)")